In [ ]:
# Evaluation

import time
import pandas as pd
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col
from sklearn.linear_model import LogisticRegression as SkLearnLR

print(">>> Mounting Google Drive...")
drive.mount('/content/drive')

spark = SparkSession.builder \
    .appName("HIGGS_Evaluation") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.memory", "6g") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 1. Load features for Scikit test and predictions
features_input_path = "/content/drive/MyDrive/Bigdata/HIGGS_features.parquet"
preds_input_path = "/content/drive/MyDrive/Bigdata/HIGGS_best_predictions.parquet"

df_scaled = spark.read.parquet(features_input_path)
best_predictions = spark.read.parquet(preds_input_path)

# STEP 5: SCALABILITY ANALYSIS

print("\n>>> STEP 5: Scalability Test PySpark vs Scikit-Learn")

sample_frac = 0.01
pdf_sample = df_scaled.sample(fraction=sample_frac, seed=42).toPandas()

if not pdf_sample.empty:
    X_sk = pdf_sample["features"].apply(lambda x: x.toArray()).tolist()
    y_sk = pdf_sample["label"].tolist()

    start = time.time()
    try:
        SkLearnLR(max_iter=5).fit(X_sk, y_sk)
        sk_time = time.time() - start
        print(f"   Scikit-Learn Time (on tiny 1% subset): {sk_time:.4f} seconds")
        print("   (Note: Scikit-Learn would crash/OOM if given the full dataset)")
    except Exception as e:
        print(f"   Scikit-Learn Error: {e}")

# STEP 6: VISUALIZATION EXPORT PARQUET

print("\n>>> STEP 6: Generating Tableau Parquet Extract")

if best_predictions is not None:
    df_with_array = best_predictions.withColumn("features_array", vector_to_array("features"))

    df_tableau_prep = df_with_array \
        .withColumn("feature_1", col("features_array")[0]) \
        .withColumn("feature_2", col("features_array")[1]) \
        .withColumn("feature_22", col("features_array")[21])

    tableau_cols = ["label", "prediction", "feature_1", "feature_2", "feature_22"]


    output_path = "/content/drive/MyDrive/Bigdata/HIGGS_tableau_extract.parquet"
    df_tableau_prep.select(*tableau_cols).coalesce(1).write.mode("overwrite").parquet(output_path)
    print(f"SUCCESS: Tableau Parquet file saved at: {output_path}")

# CSV
print("Converting Parquet to CSV for backup...")
parquet_path = "/content/drive/MyDrive/Bigdata/HIGGS_tableau_extract.parquet"
df_convert = pd.read_parquet(parquet_path)
csv_path = "/content/drive/MyDrive/Bigdata/HIGGS_tableau_extract.csv"
df_convert.to_csv(csv_path, index=False)
print(f"Tableau CSV backup ready at: {csv_path}")